# YouTube Data Extraction (Part 2): Video Metadata & Statistics

## 1. Project Overview
This second stage of the pipeline focuses on extracting detailed video-level data from the previously discovered educational channels. Instead of just channel-level metrics, we are now diving into individual video performance, including views, duration, and publication dates.

### 1.1. Environment Setup & Authentication
We install the necessary Google API client and import libraries for data manipulation and concurrent processing.

In [4]:
!pip install google-api-python-client
import googleapiclient.discovery
from googleapiclient.errors import HttpError
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
import time



# --- Configuration ---
# Replace with your secure API Key
API_KEY = "YOUR_API_KEY_HERE"

# Load the unique channel IDs collected in Part 1
df_channels = pd.read_csv("eucation_channel_list.csv")
channel_ids = df_channels["channel_id"].unique().tolist()


## 2. Extraction Logic & Functions

## 2.1. Individual Video Retrieval

The function get_single_channel_videos targets the "Uploads" playlist of a specific channel to extract the 50 most recent videos and their corresponding statistics.

In [5]:
def get_single_channel_videos(channel_data, api_key, crawling_date):
    """
    Fetches detailed video statistics for a single YouTube channel.
    """
    connection = googleapiclient.discovery.build("youtube", "v3", developerKey=api_key)
    local_videos = []
    channel_id = channel_data.get('id', 'Unknown ID')
    
    try:
        # Extract channel-level statistics
        ch_st = channel_data['statistics']
        ch_views = ch_st['viewCount']
        ch_sub = ch_st['subscriberCount']
        ch_videos = ch_st['videoCount']
        uploads_id = channel_data['contentDetails']['relatedPlaylists']['uploads']

        if not uploads_id:
            return []
            
        ch_start = channel_data['snippet']['publishedAt']
        
        # Get the list of video IDs from the 'Uploads' playlist
        request = connection.playlistItems().list(
                part="contentDetails",
                playlistId=uploads_id,
                maxResults=50
            )
        response = request.execute()
        v_ids = [item['contentDetails']['videoId'] for item in response['items']]
    
        if v_ids:
            # Fetch detailed stats for each identified video ID
            for j in range(0, len(v_ids), 50):
                v_request = connection.videos().list(
                    part="snippet,statistics,contentDetails",
                    id=",".join(v_ids[j:j+50])
                    )
                v_stat = v_request.execute()
                for v in v_stat.get('items', []):
                    details = v.get('contentDetails', {})
                    duration = details.get('duration', 'N/A')
                    local_videos.append({
                        "ch_id": channel_id,
                        "crawl": crawling_date,
                        "ch_start": ch_start,
                        "ch_sub": ch_sub,
                        "ch_views": ch_views,
                        "ch_videos": ch_videos,
                        "v_views": int(v['statistics'].get('viewCount', 0)),
                        "v_date": v['snippet']['publishedAt'],
                        "comments": int(v['statistics'].get('commentCount', 0)),
                        "v_duration": duration 
                    })
    except HttpError as e:
        if e.resp.status == 404:
            print(f"Skipping: Channel {channel_id} has no accessible public uploads.")
        else:
            print(f"API Error for {channel_id}: {e}")
        return []
    except Exception as e:
        print(f"Unexpected error for {channel_id}: {e}")
        return []
        
    return local_videos

### 2.2. Parallel Processing Implementation
To handle large volumes of data efficiently, we implement ***ThreadPoolExecutor***. This allows the script to process 10 channels simultaneously, significantly reducing total execution time.

In [6]:
def get_channels_info_fast(channel_ids, api_key):
    """
    Orchestrates the data collection using multithreading for speed.
    """
    crawl_now = datetime.now()
    crawling_date = crawl_now.strftime("%Y-%m-%d")
    
    main_conn = googleapiclient.discovery.build("youtube", "v3", developerKey=api_key)
    all_videos = []
    all_channels_data = []
    
    # 1. Fetch general channel info in batches of 50
    for i in range(0, len(channel_ids), 50):
        gap = channel_ids[i:i+50]
        res = main_conn.channels().list(
            part="snippet,contentDetails,statistics",
            id=",".join(gap)
        ).execute()
        all_channels_data.extend(res.get('items', []))
        
    # 2. Launch playlist requests in PARALLEL (10 concurrent workers)
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(executor.map(lambda c: get_single_channel_videos(c, api_key, crawling_date), all_channels_data))
    
    # Flatten the list of lists into a single list
    for res_list in results:
        all_videos.extend(res_list)
        
    return all_videos
    

## 3. Execution and Storage

## 3.1. Running the Pipeline

We now trigger the extraction and store the results in a final DataFrame.

In [7]:
              
data_all_v=get_channels_info_fast(channel_ids,API_KEY)
df_videos=pd.DataFrame(data_all_v)
df_videos.head(100)

,ch_id,crawl,ch_start,ch_sub,ch_views,ch_videos,v_views,v_date,comments,v_duration
0,UCNIVk11yBIGEUftvgeKr5yg,2026-03-27,2022-09-17T11:52:15.908936Z,127,25701,126,94,2025-01-30T18:04:56Z,0,PT23S
1,UCNIVk11yBIGEUftvgeKr5yg,2026-03-27,2022-09-17T11:52:15.908936Z,127,25701,126,16,2023-10-27T11:49:58Z,0,PT43S
2,UCNIVk11yBIGEUftvgeKr5yg,2026-03-27,2022-09-17T11:52:15.908936Z,127,25701,126,1690,2023-10-25T01:21:36Z,0,PT59S
3,UCNIVk11yBIGEUftvgeKr5yg,2026-03-27,2022-09-17T11:52:15.908936Z,127,25701,126,111,2023-07-20T19:41:48Z,2,PT11S
4,UCNIVk11yBIGEUftvgeKr5yg,2026-03-27,2022-09-17T11:52:15.908936Z,127,25701,126,10,2023-07-04T13:14:13Z,0,PT11S
...,...,...,...,...,...,...,...,...,...,...
95,UC2Zs9v2hL2qZZ7vsAENsg4w,2026-03-27,2020-03-19T11:49:39.532054Z,2060000,108364249,845,969499,2025-04-25T16:34:50Z,897,PT17M28S
96,UC2Zs9v2hL2qZZ7vsAENsg4w,2026-03-27,2020-03-19T11:49:39.532054Z,2060000,108364249,845,103682,2025-04-18T08:37:45Z,210,PT35M43S
97,UC2Zs9v2hL2qZZ7vsAENsg4w,2026-03-27,2020-03-19T11:49:39.532054Z,2060000,108364249,845,1373461,2025-04-04T06:07:45Z,1251,PT22M2S
98,UC2Zs9v2hL2qZZ7vsAENsg4w,2026-03-27,2020-03-19T11:49:39.532054Z,2060000,108364249,845,0,2025-03-28T21:38:53Z,0,P0D


In [8]:
df_videos.to_csv("educ_videos.csv", index=False, encoding='utf-8-sig')